# Tutorial: MS008 Graph Enrichment Tutorial

Audience:
- Developers extending the GTFS Visualizer graph pipeline.

Prerequisites:
- The repository virtual environment exists at `.venv`.
- You can run the CLI from the repository root.

Learning goals:
- Generate base graph artifacts.
- Run `graph-enrich` without mutating upstream artifacts.
- Inspect deterministic `route_serves_stop` enrichment edges.


## Outline

1. Create graph and graph-index artifacts
2. Run `graph-enrich`
3. Inspect `graph_enrichment_edges.json`
4. Review the shared-stop fixture behavior


In [ ]:
from __future__ import annotations

import json
import shutil
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
OUTPUT_DIR = REPO_ROOT / ".tmp" / "ms008-notebook"
FIXTURE_DIR = REPO_ROOT / "sample-data" / "fixtures" / "shared-stop-feed"

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

str(OUTPUT_DIR)


## Step 1 - Create graph and graph-index artifacts

This uses the existing ingest and graph commands first, because MS008 enrichment is strictly additive and depends on pre-existing graph indexes.


In [ ]:
subprocess.run([sys.executable, "-m", "gtfs_visualizer.cli", "ingest", str(FIXTURE_DIR), "--output-dir", str(OUTPUT_DIR)], cwd=REPO_ROOT, check=True)
subprocess.run([sys.executable, "-m", "gtfs_visualizer.cli", "graph", str(OUTPUT_DIR)], cwd=REPO_ROOT, check=True)
sorted(path.name for path in OUTPUT_DIR.iterdir())


## Step 2 - Run graph enrichment

The `graph-enrich` command writes only `graph_enrichment_edges.json`. It does not regenerate graph artifacts or alter graph-read behavior.


In [ ]:
subprocess.run([sys.executable, "-m", "gtfs_visualizer.cli", "graph-enrich", str(OUTPUT_DIR)], cwd=REPO_ROOT, check=True)
sorted(path.name for path in OUTPUT_DIR.iterdir())


## Step 3 - Inspect the enrichment edges

Notice that `trip_count` counts unique trip nodes only. The shared-stop fixture revisits `STOP1` within `T2`, but that repeated visit still contributes only one trip toward the final route-stop count.


In [ ]:
payload = json.loads((OUTPUT_DIR / "graph_enrichment_edges.json").read_text(encoding="utf-8"))
payload


## Exercises

- Delete one graph-index artifact and confirm that `graph-enrich` fails fast.
- Switch the fixture to `minimal-static-feed` and compare the smaller enrichment edge set.
- Compare `graph_edges.json` and `graph_enrichment_edges.json` to see how MS008 stays additive.
